In [0]:

# Databricks notebook source
# COMMAND ----------
# Project: Retail Intelligence Lakehouse
# Stage: Bronze ingestion
# Purpose: Create and persist raw retail source tables

# COMMAND ----------
# Cell 1: Check Spark

print("Spark version:", spark.version)

catalog = spark.sql(
    "SELECT current_catalog()"
).first()[0]

print("Current catalog:", catalog)

# COMMAND ----------
# Cell 2: Create project schema

schema_name = "retail_lakehouse"

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"`{catalog}`.`{schema_name}`"
)
    
# COMMAND ----------
# Cell 3: Create raw order records

orders_data = [
    ("O1001", "2026-10-01", "C001", "P101", 2, 499.0, "Completed", "West"),
    ("O1002", "2026-10-01", "C002", "P102", 1, 999.0, "Completed", "South"),
    ("O1003", "2026-10-02", "C003", "P103", 3, 199.0, "Completed", "North"),
    ("O1004", "2026-10-02", "C001", "P102", 1, 999.0, "Pending", "West"),
    ("O1005", "2026-10-03", None, "P101", 2, 499.0, "Completed", "East"),
    ("O1006", "2026-10-03", "C004", "P104", -1, 299.0, "Completed", "North"),
    ("O1007", "2026-10-04", "C002", "P103", 5, 199.0, "Cancelled", "South"),
    ("O1008", "2026-10-04", "C005", "P104", 2, 299.0, "Completed", "West"),
    ("O1009", "2026-10-05", "C003", "P101", 1, 499.0, "Completed", "North"),
    ("O1010", "2026-10-05", "C004", "P102", 2, 999.0, "Pending", "East"),
    ("O1003", "2026-10-02", "C003", "P103", 3, 199.0, "Completed", "North")
]

columns = [
    "order_id", "order_date", "customer_id", "product_id",
    "quantity", "unit_price", "status", "region"
]

orders_df = spark.createDataFrame(orders_data, columns)

display(orders_df)

# COMMAND ----------
# Cell 4: Create product records

products_data = [
    ("P101", "Wireless Mouse", "Accessories", 499.0),
    ("P102", "Mechanical Keyboard", "Accessories", 999.0),
    ("P103", "USB-C Hub", "Electronics", 199.0),
    ("P104", "Laptop Stand", "Accessories", 299.0)
]

product_columns = [
    "product_id", "product_name", "category", "standard_price"
]

products_df = spark.createDataFrame(
    products_data, product_columns
)

display(products_df)

# COMMAND ----------
# Cell 5: Write Delta tables

orders_table = f"{catalog}.{schema_name}.bronze_orders"
products_table = f"{catalog}.{schema_name}.bronze_products"

orders_df.write.format("delta").mode("overwrite").saveAsTable(
    orders_table
)

products_df.write.format("delta").mode("overwrite").saveAsTable(
    products_table
)

# COMMAND ----------
# Cell 6: Validate persisted data

saved_orders = spark.table(orders_table)
saved_products = spark.table(products_table)

assert saved_orders.count() == 11
assert saved_products.count() == 4

print("Bronze ingestion validation passed.")
print("Orders:", saved_orders.count())
print("Products:", saved_products.count())

display(saved_orders)
display(saved_products)


In [0]:
spark.sql(f"""
    SELECT
        'bronze_orders' AS table_name,
        COUNT(*) AS record_count
    FROM `{catalog}`.`retail_lakehouse`.`bronze_orders`

    UNION ALL

    SELECT
        'bronze_products' AS table_name,
        COUNT(*) AS record_count
    FROM `{catalog}`.`retail_lakehouse`.`bronze_products`
""").show()

In [0]:
orders = spark.table(
    f"{catalog}.retail_lakehouse.bronze_orders"
)

from pyspark.sql import functions as F

print("Total rows:", orders.count())

print(
    "Duplicate order IDs:",
    orders.groupBy("order_id")
          .count()
          .filter(F.col("count") > 1)
          .count()
)

print(
    "Missing customer IDs:",
    orders.filter(F.col("customer_id").isNull()).count()
)

print(
    "Invalid quantities:",
    orders.filter(F.col("quantity") <= 0).count()
)